# L11 — Data Analysis and Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L11/L11_starter.ipynb)

**Goal:** take a wearable-micro:bit recording with several labeled movements (still / walk / jump) and train a small machine-learning model that can tell them apart.

**Where to run this:** Google Colab (no install). Click the badge above, or *File → Open notebook → GitHub* in Colab.

**How to use this notebook:** press **`Shift + Enter`** on each cell, top to bottom. We'll do the whole pipeline once on a bundled sample recording (Part 1), then again on *your* recording (Part 2), then a bonus ML demo on handwritten digits (Part 3).

## Step 0 — Imports and the CSV loader

The micro:bit v2 datalogger writes a clean comma-separated file with a single `t,x,y,z` header. `load_microbit_log_csv(...)` is a thin wrapper around `pd.read_csv` that also computes the magnitude column for you.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def load_microbit_log_csv(path_or_url):
    """Read a micro:bit datalogger CSV. Returns a DataFrame with t, x, y, z, mag."""
    df = pd.read_csv(path_or_url)
    df = df.rename(columns={c: c.strip().lower() for c in df.columns})
    df["mag"] = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2)
    return df[["t", "x", "y", "z", "mag"]]

print("running in Colab" if IN_COLAB else "running locally")

# Part 1 — Walk through the sample recording

Before using your own data, let's run the whole pipeline on a bundled 35-second recording with labeled still/walk/jump segments.

## Step 1 — Load the sample CSV

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-still-walk-jump.csv"

if IN_COLAB:
    df = load_microbit_log_csv(SAMPLE_URL)
else:
    df = load_microbit_log_csv("../data/sample-still-walk-jump.csv")

print(df.shape)
df.head()

## Step 2 — Plot magnitude over time

You should see three different "shapes": flat (still), wiggle (walk), tall spikes (jump).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag"], color="purple")
ax.axhline(1000, linestyle="--", color="gray", label="rest")
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 3 — Label each sample

Use the segment timestamps from your capture sticky note. For the bundled sample, the schedule was:

| time (s)  | label  |
|-----------|--------|
| 0–5       | still  |
| 5–15      | walk   |
| 15–20     | still  |
| 20–30     | jump   |
| 30–35     | still  |

In [ ]:
def label_for(t):
    if t < 5:   return "still"
    if t < 15:  return "walk"
    if t < 20:  return "still"
    if t < 30:  return "jump"
    return "still"

df["label"] = df["t"].apply(label_for)
df["label"].value_counts()

In [ ]:
# Re-plot magnitude colored by label so you can SEE the classes the model will learn.
fig, ax = plt.subplots(figsize=(10, 3))
for lbl, color in [("still", "gray"), ("walk", "steelblue"), ("jump", "crimson")]:
    sub = df[df["label"] == lbl]
    ax.scatter(sub["t"], sub["mag"], s=6, color=color, label=lbl)
ax.set_xlabel("time (s)")
ax.set_ylabel("|a| (mg)")
ax.legend()
plt.show()

## Step 4 — Window the signal and extract features

A single sample (one row) doesn't tell you much. Movement lives in *patterns over time*, so we chop the recording into 1-second windows (20 samples at 20 Hz) and summarize each window with a handful of numbers — its **features**.

In [ ]:
WINDOW_S = 1.0
HZ = 20
W = int(WINDOW_S * HZ)

rows = []
for start in range(0, len(df) - W, W):
    chunk = df.iloc[start:start + W]
    rows.append({
        "mean_mag": chunk["mag"].mean(),
        "std_mag":  chunk["mag"].std(),
        "max_mag":  chunk["mag"].max(),
        "range_x":  chunk["x"].max() - chunk["x"].min(),
        "range_y":  chunk["y"].max() - chunk["y"].min(),
        "range_z":  chunk["z"].max() - chunk["z"].min(),
        "label":    chunk["label"].mode().iloc[0],
    })

features_df = pd.DataFrame(rows)
print(features_df.shape, "windows")
features_df.head()

In [ ]:
# How many examples per class? (If one class is much smaller, the model will struggle on it.)
features_df["label"].value_counts()

## Step 5 — Train/test split

We **never** train and test on the same data — that measures memorization, not learning. Hold out 30% for the test set.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["mean_mag", "std_mag", "max_mag", "range_x", "range_y", "range_z"]
X = features_df[FEATURES]
y = features_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(len(X_train), "training examples")
print(len(X_test),  "test examples")

## Step 6 — Fit a decision tree

A **decision tree** is a flowchart of `if feature > threshold` questions. We're keeping it shallow (`max_depth=3`) so the tree is small enough to read.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

print(f"Train accuracy: {clf.score(X_train, y_train):.3f}")
print(f"Test  accuracy: {clf.score(X_test,  y_test):.3f}")

## Step 7 — Read the rules the model learned

This is the payoff. The tree will look something like:

```
std_mag <= 412 ?
    yes -> still
    no  -> max_mag <= 1850 ?
               yes -> walk
               no  -> jump
```

Compare this to the hand-tuned threshold from L10 (`if mag > 2200: it's a jump`). The model **rediscovered a similar rule on its own** — and added more nuance besides. That *is* the learning.

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(clf,
          feature_names=FEATURES,
          class_names=clf.classes_,
          filled=True, rounded=True, fontsize=10, ax=ax)
plt.show()

## Step 8 — Confusion matrix

A single accuracy number hides *which* classes the model confuses. The confusion matrix shows true labels (rows) vs predicted labels (columns) — the diagonal is correct, off-diagonal is mistakes.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    clf, X_test, y_test, cmap="Blues")
plt.show()

# Part 2 — Now do the same on *your* recording

1. **Capture** a recording with the wearable micro:bit datalogger (see L11 slides for the program + procedure).
2. **Upload** the CSV with the cell below — this replaces the sample `df` with your data.
3. **Edit `label_for(t)`** to match the segment times from *your* capture sticky note.
4. Re-run the rest of the cells (`Shift + Enter`) — the pipeline is identical.

## Step 1 (your recording) — Upload and load

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()        # opens a file picker
    MY_FILE = next(iter(uploaded))
else:
    MY_FILE = "../data/sample-still-walk-jump.csv"

df = load_microbit_log_csv(MY_FILE)
print(df.shape)
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(df["t"], df["mag"], color="purple")
ax.set_xlabel("time (s)"); ax.set_ylabel("|a| (mg)")
plt.show()

## Step 2 (your recording) — Edit `label_for` to match your sequence

Change the times below to match the segment boundaries from your sticky note. Add new `if` branches if you recorded extra classes (`spin`, `wave`, etc.).

In [ ]:
def label_for(t):
    # EDIT ME: replace these times and labels with your own.
    if t < 5:   return "still"
    if t < 15:  return "walk"
    if t < 20:  return "still"
    if t < 30:  return "jump"
    return "still"

df["label"] = df["t"].apply(label_for)
df["label"].value_counts()

## Step 3 (your recording) — Window, train, evaluate (one combined cell)

Same code as Part 1, all in one cell so you can iterate fast.

In [ ]:
WINDOW_S = 1.0
HZ = 20
W = int(WINDOW_S * HZ)

rows = []
for start in range(0, len(df) - W, W):
    chunk = df.iloc[start:start + W]
    rows.append({
        "mean_mag": chunk["mag"].mean(),
        "std_mag":  chunk["mag"].std(),
        "max_mag":  chunk["mag"].max(),
        "range_x":  chunk["x"].max() - chunk["x"].min(),
        "range_y":  chunk["y"].max() - chunk["y"].min(),
        "range_z":  chunk["z"].max() - chunk["z"].min(),
        "label":    chunk["label"].mode().iloc[0],
    })
features_df = pd.DataFrame(rows)

FEATURES = ["mean_mag", "std_mag", "max_mag", "range_x", "range_y", "range_z"]
X = features_df[FEATURES]
y = features_df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)
print(f"Train accuracy: {clf.score(X_train, y_train):.3f}")
print(f"Test  accuracy: {clf.score(X_test,  y_test):.3f}")

ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap="Blues")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(clf, feature_names=FEATURES, class_names=clf.classes_,
          filled=True, rounded=True, fontsize=10, ax=ax)
plt.show()

# Part 3 — Bonus: handwritten digit recognition

The whole point of ML is that the **recipe doesn't change** when the data does. Watch: same `train_test_split`, same `.fit`, same `.score` — but now we're classifying tiny pictures of digits instead of windows of accelerometer data.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("images:", digits.images.shape, "   labels:", digits.target.shape)

# Always look at your data first.
fig, axes = plt.subplots(2, 5, figsize=(8, 3))
for ax, img, lbl in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(lbl); ax.axis("off")
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression

X = digits.images.reshape(len(digits.images), -1)   # flatten 8x8 -> 64 features
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=2000).fit(X_train, y_train)
print(f"Test accuracy: {clf.score(X_test, y_test):.3f}")

ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap="Blues")
plt.show()

**Notice what changed and what didn't:**

- *Different data.* For movement, each example was a **1-second window** of accelerometer samples summarized into **6 hand-picked numbers** (`mean_mag`, `std_mag`, `max_mag`, `range_x`, `range_y`, `range_z`) — *we* chose what to compute about each window. For digits, each example is an **8×8 grayscale image**, and the features are just the **64 raw pixel intensities** flattened into a row — no summarizing, just dump every pixel in. Both work: the model only needs *some* numbers describing each example; it doesn't care whether you engineered them or not.
- *Different model:* `LogisticRegression` instead of `DecisionTreeClassifier`.
- *Same four lines:* `train_test_split`, `.fit`, `.score`, confusion matrix.

Once you can describe an example with numbers and tag it with an answer, you can train a model on it. Movement classification and digit recognition use the same workflow.

# Reflect (homework)

Write 3–4 sentences in this cell:

- What movement classes did you record (and how many seconds of each)?
- What test accuracy did your decision tree get?
- Which two classes does the model confuse most? Why might that be?
- If you had another week, what would you change to get better accuracy?

*Your answer here.*

---

**Save your work:** *File → Download .ipynb* before the Colab runtime times out, and bring the file to L12.